# 24ADI204 – Data Science & Visualization
## Ames Housing Project: Comprehensive End-to-End Analysis (Weeks 1 to 7)

**Team:** Team 8  
**Members:** Tarun (25BAD123), Santhosh P (25BAD101), Siva Chandran (25BAD108), Santhosh M (25BAD100)  
**Dataset:** Ames Housing Dataset (Iowa, USA)  
**Academic Year:** 2026  

---

### Coursework Roadmap & Syllabus Mapping
- **Week 1 (Unit 1):** Team Formation & Data Hunting — Problem framing, acquiring dataset, baseline dimensions.
- **Week 2 (Unit 1 Practical):** "Know Your Data" — Preliminary inspection, data types, missing value profiling, target distribution.
- **Week 3 (Unit 2):** The Cleaning Sprint — Domain-specific imputation, outlier detection, dropping 200 invalid rows.
- **Week 4 (Unit 2 Practical):** EDA Deep Dive — Univariate & bivariate distributions, neighborhood trends, correlation heatmap.
- **Week 5 (Mid-Review):** Data Audit & Initial Statistical Findings — Verification, dispersion metrics, skewness analysis (+1.74).
- **Week 6 (Unit 3):** Feature Engineering — 5 composite features, log1p transformation, scaling comparisons, ordinal & one-hot encoding (293 columns).
- **Week 7 (Unit 4):** Dimensionality Reduction — VarianceThreshold, RF importance, PCA decomposition, variance visualization (Scree plot, 95% threshold).

---


## 0. Global Setup & Library Imports

Importing the core data science libraries for manipulation, visualization, statistical modeling, and dimensionality reduction.


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold, f_regression, SelectKBest
from sklearn.ensemble import RandomForestRegressor

# Visualization configuration
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

print("Libraries imported successfully.")


Libraries imported successfully.


---
# Week 1 – Team Formation & Data Hunting (Unit 1)

### Objectives
1. Acquire the raw, uncleaned Ames Housing dataset.
2. Inspect the dataset dimensions, structure, and baseline missing data profile.
3. Formulate the core analytical problem: predicting property `SalePrice` based on physical, locational, and quality attributes.


In [2]:
# Load raw dataset (Week 1 baseline)
raw_path = "Week 1/AmesHousing_Uncleaned.xlsx"
if not os.path.exists(raw_path):
    raw_path = "Week 3/AmesHousing_Uncleaned.csv"

if raw_path.endswith(".xlsx"):
    df_raw = pd.read_excel(raw_path)
else:
    df_raw = pd.read_csv(raw_path)

print("Week 1 - Raw Dataset Loaded Successfully.")
print(f"Total Rows: {df_raw.shape[0]:,}")
print(f"Total Columns: {df_raw.shape[1]}")
print(f"Total Data Cells: {df_raw.size:,}")
total_missing_cells = df_raw.isnull().sum().sum()
print(f"Total Missing Cells: {total_missing_cells:,} ({total_missing_cells / df_raw.size * 100:.2f}%)")


Week 1 - Raw Dataset Loaded Successfully.
Total Rows: 3,130
Total Columns: 82
Total Data Cells: 256,660
Total Missing Cells: 17,010 (6.63%)


---
# Week 2 – "Know Your Data" & Preliminary EDA (Unit 1 Practical)

### Objectives
1. Profile variable datatypes (numerical, categorical, ordinal).
2. Inspect the distribution of the target variable `SalePrice`.
3. Identify top features exhibiting high missing data rates.


In [3]:
# Inspect Missing Values across columns
missing_counts = df_raw.isnull().sum()
top_missing = missing_counts[missing_counts > 0].sort_values(ascending=False).head(10)
print("Top 10 Features with Missing Values in Raw Dataset:")
for col, val in top_missing.items():
    print(f" - {col:15s}: {val:4d} missing ({val / len(df_raw) * 100:.1f}%)")


Top 10 Features with Missing Values in Raw Dataset:
 - Pool QC        : 3117 missing (99.6%)
 - Misc Feature   : 3020 missing (96.5%)
 - Alley          : 2921 missing (93.3%)
 - Fence          : 2516 missing (80.4%)
 - Fireplace Qu   : 1530 missing (48.9%)
 - Lot Frontage   :  511 missing (16.3%)
 - Garage Qual    :  167 missing (5.3%)
 - Garage Cond    :  167 missing (5.3%)
 - Garage Finish  :  167 missing (5.3%)
 - Garage Yr Blt  :  167 missing (5.3%)


In [4]:
# Preliminary visualization of SalePrice distribution
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(df_raw["SalePrice"].dropna(), kde=True, color="#1f77b4", ax=ax)
ax.set_title(f"Week 2: Raw SalePrice Distribution (Skewness: {df_raw['SalePrice'].skew():.2f})", fontsize=12, fontweight="bold")
ax.set_xlabel("SalePrice ($)")
plt.tight_layout()
plt.show()


---
# Week 3 – The Cleaning Sprint & Outlier Treatment (Unit 2)

### Objectives
1. Distinguish between random missingness and structural non-presence (e.g., absence of a garage or pool).
2. Apply **domain-specific imputation**: fill categorical amenity nulls with `"None"` and numeric area nulls with `0`.
3. Filter out 200 non-standard / abnormal records (partial sales, extreme physical anomalies), producing the clean **2,930 rows × 82 columns** dataset.


In [5]:
# Load cleaned dataset produced in Week 3
clean_path = "Week 3/AmesHousing_Uncleaned_Cleaned.csv"
if not os.path.exists(clean_path):
    clean_path = "Week 5/Cleaned_AmesHousing_Week5.csv"

df_cleaned = pd.read_csv(clean_path)
print("Week 3 - Cleaned Dataset Loaded Successfully.")
print(f"Cleaned Rows: {df_cleaned.shape[0]:,}")
print(f"Cleaned Columns: {df_cleaned.shape[1]}")
print(f"Rows Removed: {df_raw.shape[0] - df_cleaned.shape[0]} (6.39% of raw records)")
print(f"Remaining Missing Values in Cleaned Data: {df_cleaned.isnull().sum().sum()}")


Week 3 - Cleaned Dataset Loaded Successfully.
Cleaned Rows: 2,930
Cleaned Columns: 82
Rows Removed: 200 (6.39% of raw records)
Remaining Missing Values in Cleaned Data: 0


---
# Week 4 – EDA Deep Dive (Unit 2 Practical)

### Objectives
1. Univariate distributions of continuous attributes (`Gr Liv Area`, `Total Bsmt SF`, `Year Built`).
2. Bivariate analysis evaluating linear relationships between living space, overall quality, and `SalePrice`.
3. Correlation analysis identifying the top numerical drivers of property value.


In [6]:
# Bivariate Analysis: Gr Liv Area & Overall Qual vs SalePrice
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gr Liv Area vs SalePrice
sns.scatterplot(data=df_cleaned, x="Gr Liv Area", y="SalePrice", alpha=0.5, color="#1f77b4", ax=ax1)
ax1.set_title("Gr Liv Area vs SalePrice", fontsize=11, fontweight="bold")
ax1.set_xlabel("Above Ground Living Area (sq ft)")
ax1.set_ylabel("SalePrice ($)")

# Overall Qual vs SalePrice
sns.boxplot(data=df_cleaned, x="Overall Qual", y="SalePrice", palette="Blues", ax=ax2)
ax2.set_title("Overall Quality (1-10) vs SalePrice", fontsize=11, fontweight="bold")
ax2.set_xlabel("Overall Quality Rating")
ax2.set_ylabel("SalePrice ($)")

plt.tight_layout()
plt.show()


In [7]:
# Top Positive Correlations with SalePrice
numeric_cols = df_cleaned.select_dtypes(include=np.number).columns
corr_series = df_cleaned[numeric_cols].corr()["SalePrice"].sort_values(ascending=False)
print("Week 4 - Top 10 Numerical Predictors Correlated with SalePrice:")
print(corr_series.iloc[1:11].round(4))


Week 4 - Top 10 Numerical Predictors Correlated with SalePrice:
Overall Qual      0.7993
Gr Liv Area       0.7068
Garage Cars       0.6476
Garage Area       0.6404
Total Bsmt SF     0.6323
1st Flr SF        0.6217
Year Built        0.5584
Full Bath         0.5456
Year Remod/Add    0.5329
Garage Yr Blt     0.5268
Name: SalePrice, dtype: float64


---
# Week 5 – Mid-Review: Data Audit & Initial Statistical Findings

### Objectives
1. Perform formal quality verification: 2,930 verified records with 0 nulls and 0 duplicates.
2. Evaluate descriptive statistics (mean, median, standard deviation, IQR) to assess data dispersion.
3. Formally quantify skewness in `SalePrice` (+1.74) and motivate the need for normalization prior to linear modeling.


In [8]:
# Descriptive Statistics and Dispersion Summary
audit_vars = ["SalePrice", "Gr Liv Area", "Total Bsmt SF", "Garage Area", "Year Built"]
stats_summary = []
for v in audit_vars:
    mean_val = df_cleaned[v].mean()
    median_val = df_cleaned[v].median()
    std_val = df_cleaned[v].std()
    iqr_val = df_cleaned[v].quantile(0.75) - df_cleaned[v].quantile(0.25)
    skew_val = df_cleaned[v].skew()
    stats_summary.append({
        "Feature": v,
        "Mean": round(mean_val, 2),
        "Median": round(median_val, 2),
        "Std Dev": round(std_val, 2),
        "IQR": round(iqr_val, 2),
        "Skewness": round(skew_val, 2)
    })

stats_df = pd.DataFrame(stats_summary)
print("Week 5 - Audit of Key Feature Distributions & Skewness:")
display(stats_df)


---
# Week 6 – Feature Engineering, Scaling & Encoding (Unit 3)

### Objectives
1. **Domain Feature Construction**: Create 5 composite physical metrics:
   - `HouseAge` = `Yr Sold` - `Year Built`
   - `RemodAge` = `Yr Sold` - `Year Remod/Add`
   - `TotalBathrooms` = `Full Bath` + $0.5 \times$ `Half Bath` + `Bsmt Full Bath` + $0.5 \times$ `Bsmt Half Bath`
   - `TotalPorchArea` = Sum of all porch square footage
   - `TotalHouseArea` = `Gr Liv Area` + `Total Bsmt SF`
2. **Skewness Mitigation**: Apply `log1p(SalePrice)`, reducing skewness from **+1.74 to +0.12** (normal bell curve).
3. **Feature Scaling**: Compare Min-Max Normalization, Standardization (Z-score), and Robust Scaling.
4. **Categorical Encoding**: Ordinal encoding for 10 quality rankings ($0 \to 5$) and One-Hot Encoding for 33 nominal categories, expanding features to **293 columns**.


In [9]:
# 1. Feature Creation
df_w6 = df_cleaned.copy()
df_w6["HouseAge"] = df_w6["Yr Sold"] - df_w6["Year Built"]
df_w6["RemodAge"] = df_w6["Yr Sold"] - df_w6["Year Remod/Add"]
df_w6["TotalBathrooms"] = df_w6["Full Bath"] + (0.5 * df_w6["Half Bath"]) + df_w6["Bsmt Full Bath"] + (0.5 * df_w6["Bsmt Half Bath"])
df_w6["TotalPorchArea"] = df_w6["Open Porch SF"] + df_w6["Enclosed Porch"] + df_w6["3Ssn Porch"] + df_w6["Screen Porch"] + df_w6["Wood Deck SF"]
df_w6["TotalHouseArea"] = df_w6["Gr Liv Area"] + df_w6["Total Bsmt SF"]

# 2. Skewness Transformation
df_w6["SalePrice_Log"] = np.log1p(df_w6["SalePrice"])
print(f"SalePrice Skewness Before: {df_w6['SalePrice'].skew():.2f}")
print(f"SalePrice Skewness After log1p: {df_w6['SalePrice_Log'].skew():.2f}")

# 3. Ordinal Encoding
qual_map = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
ord_cols = ["Exter Qual", "Exter Cond", "Bsmt Qual", "Bsmt Cond", "Heating QC", "Kitchen Qual", "Fireplace Qu", "Garage Qual", "Garage Cond", "Pool QC"]
for c in ord_cols:
    if c in df_w6.columns:
        df_w6[c + "_Enc"] = df_w6[c].map(lambda x: qual_map.get(str(x), 0))

# 4. One-Hot Encoding
nom_cols = ["MS Zoning", "Street", "Alley", "Lot Shape", "Land Contour", "Utilities", "Lot Config",
            "Land Slope", "Neighborhood", "Condition 1", "Condition 2", "Bldg Type", "House Style",
            "Roof Style", "Roof Matl", "Exterior 1st", "Exterior 2nd", "Mas Vnr Type", "Foundation",
            "Bsmt Exposure", "BsmtFin Type 1", "BsmtFin Type 2", "Heating", "Central Air", "Electrical",
            "Functional", "Garage Type", "Garage Finish", "Paved Drive", "Fence", "Misc Feature",
            "Sale Type", "Sale Condition"]

df_encoded = pd.get_dummies(df_w6, columns=[c for c in nom_cols if c in df_w6.columns], drop_first=False)
print(f"Week 6 - Feature Engineered Matrix Shape: {df_encoded.shape}")


SalePrice Skewness Before: 1.74
SalePrice Skewness After log1p: 0.12
Week 6 - Feature Engineered Matrix Shape: (2930, 293)


In [10]:
# Visualization: Target Distribution Before vs After log1p
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df_w6["SalePrice"], kde=True, color="#d62728", ax=ax1)
ax1.set_title(f"Original SalePrice (Skew: {df_w6['SalePrice'].skew():.2f})", fontsize=11, fontweight="bold")
sns.histplot(df_w6["SalePrice_Log"], kde=True, color="#2ca02c", ax=ax2)
ax2.set_title(f"Log-Transformed SalePrice (Skew: {df_w6['SalePrice_Log'].skew():.2f})", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()


---
# Week 7 – Dimensionality Reduction: Feature Selection & PCA (Unit 4)

### Objectives
1. Address the high-dimensional feature explosion (293 columns) resulting from Week 6.
2. Implement multiple **Feature Selection** strategies:
   - **Low-Variance Filter** (`VarianceThreshold = 0.01`): Pruning 88 quasi-constant binary features.
   - **Correlation & Collinearity Pruning**: Removing redundant collinear predictor pairs ($r > 0.85$).
   - **Univariate Selection & Random Forest Importance**: Identifying top drivers of property valuation.
   - Export curated subset (`Selected_Features_AmesHousing.csv`).
3. Implement **Principal Component Analysis (PCA)**:
   - Standardize all 279 numerical features (`StandardScaler`).
   - Analyze explained variance and plot Scree / Cumulative curves (80%, 90%, 95% retention thresholds).
   - 2D PCA projection colored by property value.
   - Interpret principal component loadings (eigenvectors).
   - Export transformed dataset (`PCA_Transformed_AmesHousing.csv`).


In [11]:
# Prepare Feature Space for Week 7
raw_cat_cols = df_encoded.select_dtypes(include=['object', 'string']).columns.tolist()
df_w7_clean = df_encoded.drop(columns=raw_cat_cols)

id_cols = ['Order', 'PID']
target_cols = ['SalePrice', 'SalePrice_Log']
feature_cols = [c for c in df_w7_clean.columns if c not in id_cols + target_cols]

X = df_w7_clean[feature_cols].astype(float)
y = df_w7_clean['SalePrice_Log']

print(f"Predictor Feature Space Shape: {X.shape}")
print(f"Target Observations: {len(y)}")


Predictor Feature Space Shape: (2930, 279)
Target Observations: 2930


In [12]:
# 1. Feature Selection: Variance Threshold
vt = VarianceThreshold(threshold=0.01)
vt.fit(X)
vt_kept_cols = [col for col, supp in zip(feature_cols, vt.get_support()) if supp]
print(f"Initial Features: {len(feature_cols)}")
print(f"Dropped Quasi-Constant Features: {len(feature_cols) - len(vt_kept_cols)}")
print(f"Features Retained (Variance >= 0.01): {len(vt_kept_cols)}")

# 2. Random Forest Importance & Univariate F-Scores
rf = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X[vt_kept_cols], y)
rf_imp = pd.Series(rf.feature_importances_, index=vt_kept_cols).sort_values(ascending=False)

selector_f = SelectKBest(score_func=f_regression, k=40)
selector_f.fit(X[vt_kept_cols], y)
f_scores = pd.Series(selector_f.scores_, index=vt_kept_cols).sort_values(ascending=False)

print("\nTop 5 Features by Random Forest Importance:")
for col, imp in rf_imp.head(5).items():
    print(f" - {col:18s}: {imp * 100:.2f}%")


Initial Features: 279
Dropped Quasi-Constant Features: 88
Features Retained (Variance >= 0.01): 191

Top 5 Features by Random Forest Importance:
 - Overall Qual      : 49.73%
 - TotalHouseArea    : 28.05%
 - TotalBathrooms    : 1.90%
 - Gr Liv Area       : 1.23%
 - Lot Area          : 1.14%


In [13]:
# 3. Principal Component Analysis (PCA)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(random_state=42)
pca.fit(X_scaled)

exp_var = pca.explained_variance_ratio_
cum_var = np.cumsum(exp_var)

n_80 = int(np.argmax(cum_var >= 0.80) + 1)
n_90 = int(np.argmax(cum_var >= 0.90) + 1)
n_95 = int(np.argmax(cum_var >= 0.95) + 1)

print(f"PC1 Variance Explained: {exp_var[0]*100:.2f}%")
print(f"PC2 Variance Explained: {exp_var[1]*100:.2f}%")
print(f"Top 2 PCs Combined Variance: {(exp_var[0] + exp_var[1])*100:.2f}%")
print(f"Components for 80% Variance: {n_80} (out of {X.shape[1]}) -> {(1 - n_80/X.shape[1])*100:.1f}% reduction")
print(f"Components for 90% Variance: {n_90} (out of {X.shape[1]}) -> {(1 - n_90/X.shape[1])*100:.1f}% reduction")
print(f"Components for 95% Variance: {n_95} (out of {X.shape[1]}) -> {(1 - n_95/X.shape[1])*100:.1f}% reduction")


PC1 Variance Explained: 6.90%
PC2 Variance Explained: 2.89%
Top 2 PCs Combined Variance: 9.79%
Components for 80% Variance: 117 (out of 279) -> 58.1% reduction
Components for 90% Variance: 152 (out of 279) -> 45.5% reduction
Components for 95% Variance: 175 (out of 279) -> 37.3% reduction


In [14]:
# Visualizing Variance: Scree Plot & Cumulative Explained Variance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Scree Plot (Top 25 Components)
ax1.plot(range(1, 26), exp_var[:25] * 100, marker="o", color="#1f77b4", linewidth=2)
ax1.bar(range(1, 26), exp_var[:25] * 100, alpha=0.3, color="#1f77b4")
ax1.set_xlabel("Principal Component Number")
ax1.set_ylabel("Individual Explained Variance (%)")
ax1.set_title("Scree Plot (Elbow Criterion)", fontsize=11, fontweight="bold")
ax1.grid(True, linestyle="--", alpha=0.6)

# Right: Cumulative Explained Variance
ax2.plot(range(1, len(cum_var) + 1), cum_var * 100, color="#d62728", linewidth=2.5, label="Cumulative Variance")
ax2.axhline(80, color="gray", linestyle="--", label=f"80% ({n_80} PCs)")
ax2.axhline(90, color="orange", linestyle="--", label=f"90% ({n_90} PCs)")
ax2.axhline(95, color="green", linestyle="--", label=f"95% ({n_95} PCs)")
ax2.scatter([n_80, n_90, n_95], [cum_var[n_80-1]*100, cum_var[n_90-1]*100, cum_var[n_95-1]*100], color="black", zorder=5)
ax2.set_xlabel("Number of Principal Components")
ax2.set_ylabel("Cumulative Explained Variance (%)")
ax2.set_title("Cumulative Explained Variance vs Components", fontsize=11, fontweight="bold")
ax2.legend(loc="lower right")
ax2.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()


In [15]:
# 2D PCA Projection colored by SalePrice
pca_2d = PCA(n_components=2, random_state=42)
X_pca_2d = pca_2d.fit_transform(X_scaled)

plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=df_w7_clean["SalePrice"], cmap="plasma", alpha=0.65, s=25)
cbar = plt.colorbar(scatter)
cbar.set_label("SalePrice ($)")
plt.xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% Variance)")
plt.ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% Variance)")
plt.title("2D PCA Projection: Ames Housing (Colored by SalePrice)", fontsize=12, fontweight="bold")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


---
# Summary of Completed Milestones (Weeks 1 to 7)

| Week | Milestone / Topic | Key Implementation | Transformation Outcome |
|:---|:---|:---|:---|
| **Week 1** | Team Formation & Data Hunting | Raw data acquisition & structural profiling | 3,130 rows × 82 features (6.63% missingness) |
| **Week 2** | "Know Your Data" | Missingness analysis & target distribution | Identified right-skewed `SalePrice` |
| **Week 3** | The Cleaning Sprint | Domain imputation & outlier detection | Filtered 200 invalid records $\to$ **2,930 clean rows** |
| **Week 4** | EDA Deep Dive | Univariate/bivariate analysis & heatmap | Identified living area & quality as top drivers |
| **Week 5** | Mid-Review & Data Audit | Descriptive statistics, dispersion & skewness | Formally measured +1.74 skewness on `SalePrice` |
| **Week 6** | Feature Engineering | 5 composite features, log1p transform, encoding | Expanded to **293 columns** ($0.12$ target skewness) |
| **Week 7** | Dimensionality Reduction | VarianceThreshold, RF selection & PCA | Pruned 88 low-var cols; 175 PCs retain 95% variance |

---
### Conclusion
Over the 7-week trajectory, the Ames Housing dataset was cleaned, explored, engineered, and compressed into optimal representations (`Selected_Features_AmesHousing.csv` with 40 curated features and `PCA_Transformed_AmesHousing.csv` with 175 orthogonal principal components). These prepared datasets provide a leak-free foundation for downstream regression modeling.
